# 🎬 ClipSense — Multi-Agent Video Intelligence (Colab + Groq, 100% free tier)

**Don't watch it all. Understand it all.**

Run the cells **top to bottom**. What you get: transcript (captions or Whisper), slide/code OCR + visual descriptions, auto chapters, key takeaways, fact-checking Critic agent, timestamped chat (RAG), flashcards, and Markdown/HTML export.

| Agent | What it does here |
|---|---|
| Orchestrator | Classifies the video, builds the plan, runs agents in parallel, handles errors |
| Ingestion | `yt-dlp` + `ffmpeg` (or your uploaded file) |
| Transcript | YouTube captions first, else Groq Whisper |
| Vision | OpenCV frame sampling + dedupe, Groq vision model for type/OCR/description |
| Audio | `librosa` emphasis / quiet detection |
| Fusion + Segmentation | One timeline, embedding-based topic boundaries |
| Summarizer | Map-reduce with timestamps |
| Critic | Verifies each claim against the transcript, feedback loop |
| Q&A | Retrieval-augmented chat with timestamp citations |

**Before you start:** get a free key at https://console.groq.com/keys, then in Colab click the 🔑 *Secrets* icon, add `GROQ_API_KEY`, and turn on *Notebook access*. (If you skip this, the notebook will ask you to paste it.)

In [1]:
# CELL 1 - Install dependencies (about 1-2 minutes)
!pip -q install -U groq yt-dlp youtube-transcript-api sentence-transformers imagehash markdown gradio
!apt-get -qq install -y ffmpeg > /dev/null 2>&1
print("Installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.3/111.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 2.7 MB/s eta 0:00:00
Installed.


In [2]:
# NEW CELL - Install PO Token plugin
!pip install -U "yt-dlp>=2025.05.22"
!pip install -U bgutil-ytdlp-pot-provider

# Verify installation
!yt-dlp --version
!pip show bgutil-ytdlp-pot-provider | grep -E "Name|Version"

2026.08.19
Name: bgutil-ytdlp-pot-provider
Version: 2.0.0


## Cell 2 — Configuration, Groq client, retry + JSON helpers

In [23]:
# CELL 2 - Setup
import os, re, json, time, base64, math, glob, shutil, subprocess, getpass
from concurrent.futures import ThreadPoolExecutor
from groq import Groq

# Get Groq API key from Google Colab Secrets
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = None

GROQ_API_KEY = (
    GROQ_API_KEY
    or os.environ.get("GROQ_API_KEY")
    or getpass.getpass("Paste your Groq API key: ")
)

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# Groq client
client = Groq(
    api_key=GROQ_API_KEY,
    max_retries=0
)

# ============================================================
# GROQ MODEL CONFIGURATION
# ============================================================

CFG = dict(
    # Main reasoning / summarization model
    # UPDATED: llama-3.3-70b-versatile was retired by Groq on 2026-08-16.
    # Replacement: openai/gpt-oss-120b (recommended by Groq).
    LLM_MODEL="openai/gpt-oss-120b",

    # Fast/cheap classification model
    # UPDATED: llama-3.1-8b-instant was also retired.
    # Replacement: openai/gpt-oss-20b.
    FAST_MODEL="openai/gpt-oss-20b",

    # Vision model
    # UPDATED: meta-llama/llama-4-scout-17b-16e-instruct was retired.
    # qwen/qwen3.8-27b is the vision-capable model on your account.
    VISION_MODEL="qwen/qwen3.8-27b",

    # Speech-to-text model
    ASR_MODEL="whisper-large-v3-turbo",

    # ========================================================
    # CLIPSENSE CONFIGURATION
    # ========================================================

    WORK_DIR="/content/clipsense",

    # Process only the first N minutes during testing
    MAX_MINUTES=10,

    # Video download resolution
    VIDEO_HEIGHT=480,

    # Maximum frames sent to vision model
    MAX_VISION_FRAMES=25,

    # Summarization chunk size
    CHUNK_WORDS=900,

    # Minimum chunk size
    MIN_CHUNK_WORDS=400,

    # Keep this low for Groq free-tier limits
    LLM_WORKERS=1,
)


# ============================================================
# TIMESTAMP HELPERS
# ============================================================

def fmt_ts(sec):
    sec = int(max(0, sec))
    h, r = divmod(sec, 3600)
    m, s = divmod(r, 60)
    return f"{h}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"


def parse_ts(x):
    """
    '12:34' or '1:02:03' or number -> seconds.
    Returns None if unparseable.
    """
    if x is None:
        return None
    if isinstance(x, (int, float)):
        return float(x)
    parts = re.findall(r"\d+", str(x))
    if not parts:
        return None
    parts = [int(p) for p in parts][-3:]
    sec = 0
    for p in parts:
        sec = sec * 60 + p
    return float(sec)


# ============================================================
# GROQ RETRY / RATE LIMIT HANDLING
# ============================================================

def parse_wait(msg, default):
    m = re.search(
        r"try again in\s+(?:(\d+)h)?(?:(\d+)m)?([\d.]+)s",
        msg
    )
    if m:
        h, mi, s = m.groups()
        return int(h or 0) * 3600 + int(mi or 0) * 60 + float(s) + 1
    return default


def with_retry(fn, retries=7, label="call"):
    """
    Retry wrapper that understands Groq 429 rate limits.
    """
    for attempt in range(retries):
        try:
            return fn()
        except Exception as e:
            msg = str(e)
            rate = (
                ("429" in msg)
                or ("rate_limit" in msg.lower())
                or ("rate limit" in msg.lower())
            )
            fatal = (
                (not rate)
                and any(
                    c in msg[:80]
                    for c in ("Error code: 400", "Error code: 401", "Error code: 404")
                )
            )
            if fatal or attempt == retries - 1:
                raise
            wait = (
                parse_wait(msg, 15 * (attempt + 1))
                if rate
                else 3 * (attempt + 1)
            )
            if rate and wait > 240:
                raise RuntimeError(
                    f"[{label}] Groq quota exhausted "
                    f"(would need to wait {wait/60:.0f} min). "
                    f"Lower CFG['MAX_MINUTES'] or retry later.\n{msg}"
                )
            print(
                f"   [{label}] "
                f"{'rate limit' if rate else 'error'} "
                f"- waiting {wait:.0f}s "
                f"(retry {attempt+1}/{retries-1})"
            )
            time.sleep(wait)


# ============================================================
# JSON HELPERS
# ============================================================

def safe_json(text):
    if not text:
        return {}
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r"\{.*\}", text, re.S)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass
    return {}


# ============================================================
# LLM FUNCTION
# ============================================================

def llm(
    messages,
    model=None,
    temperature=0.1,
    max_tokens=2000,
    json_mode=False
):
    """
    Wrapper around Groq chat completions.

    If json_mode=True and Groq's server-side JSON validator rejects the
    output (BadRequestError with 'json_validate_failed'), retry the same
    prompt in plain-text mode. safe_json() will then extract the JSON.
    """
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    def go():
        kw = (
            {"response_format": {"type": "json_object"}}
            if json_mode
            else {}
        )
        r = client.chat.completions.create(
            model=model or CFG["LLM_MODEL"],
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
            **kw
        )
        if r.choices[0].finish_reason == "length":
            print("   [llm] warning: output truncated, consider raising max_tokens")
        return r.choices[0].message.content

    try:
        return with_retry(go, label="llm")
    except Exception as e:
        if json_mode and (
            "json_validate_failed" in str(e)
            or "Failed to validate JSON" in str(e)
        ):
            print("   [llm] JSON mode failed, retrying as plain text...")
            return with_retry(
                lambda: llm(
                    messages,
                    model=model,
                    temperature=temperature,
                    max_tokens=max_tokens,
                    json_mode=False
                ),
                label="llm-fallback"
            )
        raise


def llm_json(prompt, system=None, **kw):
    msgs = (
        [{"role": "system", "content": system}]
        if system
        else []
    )
    msgs += [{"role": "user", "content": prompt}]
    return safe_json(llm(msgs, json_mode=True, **kw))


# ============================================================
# BASIC GROQ CONNECTION TEST
# ============================================================

print("Testing Groq connection...")
print(llm("Reply with the single word: ready", max_tokens=5))
print("Groq connection successful!")

Testing Groq connection...
   [llm] warning: output truncated, consider raising max_tokens

Groq connection successful!


In [24]:
print(CFG["VISION_MODEL"])

qwen/qwen3.8-27b


In [25]:
# CELL 3 - Check that the configured Groq models still exist (Groq renames/retires models sometimes)
available = {m.id for m in client.models.list().data}
for key in ("LLM_MODEL", "FAST_MODEL", "VISION_MODEL", "ASR_MODEL"):
    ok = CFG[key] in available
    print(("OK      " if ok else "MISSING ") + f"{key:13s} {CFG[key]}")
if CFG["VISION_MODEL"] not in available:
    cands = [m for m in sorted(available) if any(t in m for t in ("scout", "vision", "maverick", "llama-4"))]
    print("\nVision candidates on your account:", cands)
    print("-> set CFG['VISION_MODEL'] to one of them.")
print("\nAll available models:\n", sorted(available))

OK      LLM_MODEL     openai/gpt-oss-120b
OK      FAST_MODEL    openai/gpt-oss-20b
OK      VISION_MODEL  qwen/qwen3.8-27b
OK      ASR_MODEL     whisper-large-v3-turbo

All available models:
 ['allam-2-7b', 'canopylabs/orpheus-arabic-saudi', 'canopylabs/orpheus-v1-english', 'meta-llama/llama-prompt-guard-2-22m', 'meta-llama/llama-prompt-guard-2-86m', 'openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'openai/gpt-oss-safeguard-20b', 'qwen/qwen3.8-27b', 'whisper-large-v3', 'whisper-large-v3-turbo']


## Cell 4 — Shared memory (blackboard), message bus, base Agent class

In [5]:
# CELL 4 - Blackboard + messaging
class Blackboard:
    """Shared memory + message log that every agent reads/writes."""
    def __init__(self):
        self.mem, self.messages, self.t0 = {}, [], time.time()
    def write(self, k, v): self.mem[k] = v
    def read(self, k, default=None): return self.mem.get(k, default)
    def post(self, frm, to, task, payload=None, priority=1):
        msg = {"from": frm, "to": to, "task": task, "payload": payload, "priority": priority,
               "t": round(time.time() - self.t0, 1)}
        self.messages.append(msg)
        print(f"[{msg['t']:>7}s] {frm:>12} -> {to:<12} {task}")
        return msg

class Agent:
    name = "Agent"
    def __init__(self, bb): self.bb = bb
    def say(self, to, task, payload=None): return self.bb.post(self.name, to, task, payload)

## Cell 5 — Ingestion Agent (download video + extract audio)

⚠️ YouTube often blocks Colab's IP addresses. If the download fails, use the **upload fallback** (see the run cell) or a `cookies.txt` file. See the troubleshooting section at the bottom.

In [6]:
# CELL 5 - Ingestion Agent
import yt_dlp

def extract_video_id(url):
    m = re.search(r"(?:v=|youtu\.be/|shorts/|embed/|live/)([A-Za-z0-9_-]{11})", url or "")
    return m.group(1) if m else None

def ffprobe_duration(path):
    out = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                          "-of", "default=nw=1:nk=1", path], capture_output=True, text=True).stdout.strip()
    try: return float(out)
    except Exception: return 0.0

class IngestionAgent(Agent):
    name = "Ingestion"

    def run(self, url=None, local_file=None, cookies_file=None):
        if local_file:
            vid = "local_" + re.sub(r"\W+", "_", os.path.basename(local_file))[:40]
            d = f"{CFG['WORK_DIR']}/{vid}"; os.makedirs(d, exist_ok=True)
            video_path = local_file
            meta = {"id": vid, "title": os.path.basename(local_file), "channel": "", "description": "",
                    "duration": ffprobe_duration(local_file), "chapters": [], "url": None}
        else:
            vid = extract_video_id(url)
            if not vid: raise ValueError("Could not find a YouTube video id in that URL.")
            d = f"{CFG['WORK_DIR']}/{vid}"; os.makedirs(d, exist_ok=True)
            cached = [p for p in glob.glob(f"{d}/video.*") if not p.endswith((".part", ".ytdl"))]
            if cached and os.path.exists(f"{d}/meta.json"):
                self.say("Orchestrator", "cache hit - reusing downloaded video")
                video_path, meta = cached[0], json.load(open(f"{d}/meta.json"))
            else:
                h = CFG["VIDEO_HEIGHT"]
                opts = {"format": f"bv*[height<={h}]+ba/b[height<={h}]/b", "outtmpl": f"{d}/video.%(ext)s",
                        "merge_output_format": "mp4", "quiet": True, "no_warnings": True,
                        "noplaylist": True, "retries": 5}
                if cookies_file: opts["cookiefile"] = cookies_file
                self.say("Orchestrator", "downloading video (this can take a while for long videos)...")
                try:
                    with yt_dlp.YoutubeDL(opts) as ydl:
                        info = ydl.extract_info(url, download=True)
                except Exception as e:
                    raise RuntimeError(
                        "YouTube download failed (very common on Colab - YouTube blocks datacenter IPs).\n"
                        "Fix: (a) upload the video file and set LOCAL_FILE, or (b) upload cookies.txt and set "
                        "COOKIES_FILE. See troubleshooting at the end of the notebook.\n\nOriginal error: " + str(e))
                files = [p for p in glob.glob(f"{d}/video.*") if not p.endswith((".part", ".ytdl"))]
                video_path = max(files, key=os.path.getsize)
                meta = {"id": vid, "title": info.get("title", ""), "channel": info.get("channel") or info.get("uploader", ""),
                        "description": (info.get("description") or "")[:2000], "duration": info.get("duration") or ffprobe_duration(video_path),
                        "chapters": [{"start": c.get("start_time", 0), "title": c.get("title", "")} for c in (info.get("chapters") or [])],
                        "url": url}
                json.dump(meta, open(f"{d}/meta.json", "w"))

        limit = CFG["MAX_MINUTES"] * 60
        meta["process_seconds"] = min(meta["duration"] or limit, limit)
        audio = f"{d}/audio.mp3"
        if not os.path.exists(audio):
            subprocess.run(["ffmpeg", "-y", "-i", video_path, "-t", str(limit), "-vn", "-ac", "1", "-ar", "16000",
                            "-b:a", "32k", audio], check=True, capture_output=True)
        media = {"dir": d, "video": video_path, "audio": audio, "meta": meta}
        self.bb.write("media", media)
        self.say("Orchestrator", f"ready: '{meta['title']}' ({fmt_ts(meta['duration'])}), processing first {fmt_ts(meta['process_seconds'])}")
        return media

## Cell 6 — Transcript Agent (captions first, Groq Whisper fallback)

In [7]:
# CELL 6 - Transcript Agent
from youtube_transcript_api import YouTubeTranscriptApi

def regroup(segs, min_words=18, max_words=60):
    """Merge tiny caption/ASR pieces into sentence-like segments."""
    out, cur = [], None
    for s in segs:
        t = re.sub(r"\s+", " ", s["text"]).strip()
        if not t or re.fullmatch(r"\[.*?\]", t): continue
        if cur is None: cur = {"start": s["start"], "end": s["end"], "text": t}
        else: cur["text"] += " " + t; cur["end"] = s["end"]
        n = len(cur["text"].split())
        if (n >= min_words and re.search(r"[.!?]$", cur["text"])) or n >= max_words:
            out.append(cur); cur = None
    if cur: out.append(cur)
    return out

class TranscriptAgent(Agent):
    name = "Transcript"

    def captions(self, vid, lang_hint=None, limit=None):
        langs = ([lang_hint] if lang_hint else []) + ["en", "en-US", "en-GB", "hi", "ur"]
        api = YouTubeTranscriptApi()
        if hasattr(api, "fetch"):
            tr = api.fetch(vid, languages=langs)
            raw = [{"start": s.start, "end": s.start + s.duration, "text": s.text} for s in tr]
        else:
            tr = YouTubeTranscriptApi.get_transcript(vid, languages=langs)
            raw = [{"start": s["start"], "end": s["start"] + s["duration"], "text": s["text"]} for s in tr]
        if limit: raw = [s for s in raw if s["start"] < limit]
        return raw

    def whisper(self, audio_path, lang=None):
        d, dur, CH = os.path.dirname(audio_path), ffprobe_duration(audio_path), 600
        n, segs, detected = max(1, math.ceil(dur / CH)), [], None
        for i in range(n):
            off, part = i * CH, f"{d}/part_{i:03d}.mp3"
            subprocess.run(["ffmpeg", "-y", "-ss", str(off), "-t", str(CH), "-i", audio_path, "-c", "copy", part],
                           check=True, capture_output=True)
            def go():
                with open(part, "rb") as f:
                    kw = dict(file=(os.path.basename(part), f.read()), model=CFG["ASR_MODEL"],
                              response_format="verbose_json", temperature=0.0, timestamp_granularities=["segment"])
                    if lang: kw["language"] = lang
                    return client.audio.transcriptions.create(**kw)
            r = with_retry(go, label=f"whisper {i+1}/{n}")
            data = r if isinstance(r, dict) else (r.model_dump() if hasattr(r, "model_dump") else dict(vars(r)))
            detected = detected or data.get("language")
            got = data.get("segments") or []
            for s in got:
                if s.get("no_speech_prob", 0) > 0.9: continue
                segs.append({"start": off + float(s["start"]), "end": off + float(s["end"]), "text": s["text"]})
            if not got and data.get("text"):
                segs.append({"start": off, "end": off + CH, "text": data["text"]})
            print(f"   whisper chunk {i+1}/{n} done")
        return segs, detected

    def run(self, media, lang_hint=None, force_whisper=False):
        meta, limit = media["meta"], media["meta"]["process_seconds"]
        raw, source, lang = None, None, lang_hint
        vid = meta["id"]
        if not force_whisper and not vid.startswith("local_"):
            try:
                raw = self.captions(vid, lang_hint, limit); source = "youtube-captions"
                self.say("Orchestrator", f"using official captions ({len(raw)} lines)")
            except Exception as e:
                self.say("Orchestrator", f"no usable captions ({type(e).__name__}) -> falling back to Whisper")
        if not raw:
            raw, detected = self.whisper(media["audio"], lang_hint); source = "whisper"; lang = detected or lang
        segs = regroup(raw)
        if not segs: raise RuntimeError("Transcript is empty (no speech found?).")
        res = {"segments": segs, "source": source, "language": lang}
        self.bb.write("transcript", res)
        self.say("Fusion", f"transcript ready: {len(segs)} segments, {sum(len(s['text'].split()) for s in segs)} words ({source})")
        return res

## Cell 7 — Vision Agent (frame sampling → dedupe → Groq vision for type + OCR + description)

In [8]:
# CELL 7 - Vision Agent
import cv2, imagehash, numpy as np
from PIL import Image

VISION_PROMPT = (
    "You analyse ONE frame from a video. Reply with ONLY a JSON object with keys:\n"
    '"frame_type": one of ["slide","code_editor","chart","whiteboard","talking_head","demo","broll"],\n'
    '"ocr_text": all legible on-screen text (slide titles, bullets, code, formulas) or "" if none,\n'
    '"description": 1-2 sentences describing diagrams/charts/demos (empty string for talking_head).\n'
    "Do not add any text outside the JSON."
)

class VisionAgent(Agent):
    name = "Vision"

    def sample(self, video_path, limit_sec, every):
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS) or 25
        total = cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0
        dur = min(total / fps if total else limit_sec, limit_sec)
        fdir = os.path.join(os.path.dirname(video_path), "frames"); shutil.rmtree(fdir, ignore_errors=True); os.makedirs(fdir)
        kept, hashes, last_h, last_g, t = [], [], None, None, 0.0
        while t < dur:
            cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
            ok, frame = cap.read()
            if not ok: break
            pil = Image.fromarray(cv2.cvtColor(cv2.resize(frame, (320, 180)), cv2.COLOR_BGR2RGB))
            h = imagehash.dhash(pil, hash_size=16)
            g = cv2.cvtColor(cv2.resize(frame, (160, 90)), cv2.COLOR_BGR2GRAY).astype("float32")
            new = True if last_h is None else ((h - last_h) >= 14 or float(np.abs(g - last_g).mean()) >= 7)
            if new and any((h - k) < 8 for k in hashes): new = False   # already seen earlier (duplicate)
            if new:
                fr = frame
                if fr.shape[1] > 960: fr = cv2.resize(fr, (960, int(fr.shape[0] * 960 / fr.shape[1])))
                p = f"{fdir}/f_{int(t):06d}.jpg"; cv2.imwrite(p, fr, [cv2.IMWRITE_JPEG_QUALITY, 80])
                kept.append({"timestamp": float(t), "path": p}); hashes.append(h); last_h, last_g = h, g
            t += every
        cap.release()
        return kept

    def analyse(self, path):
        b64 = base64.b64encode(open(path, "rb").read()).decode()
        def go():
            r = client.chat.completions.create(
                model=CFG["VISION_MODEL"], temperature=0.1, max_tokens=700,
                messages=[{"role": "user", "content": [
                    {"type": "text", "text": VISION_PROMPT},
                    {"type": "image_url", "image_url": {"url": "data:image/jpeg;base64," + b64}}]}])
            return r.choices[0].message.content
        return safe_json(with_retry(go, label="vision"))

    def run(self, media, budget, every):
        limit = media["meta"]["process_seconds"]
        kept = self.sample(media["video"], limit, every)
        self.say("Orchestrator", f"{len(kept)} visually-unique frames after dedupe")
        if len(kept) > budget:
            idx = np.linspace(0, len(kept) - 1, budget).round().astype(int)
            kept = [kept[i] for i in sorted(set(idx))]
        self.say("Orchestrator", f"analysing {len(kept)} frames with {CFG['VISION_MODEL'].split('/')[-1]}")
        errors = []
        def work(f):
            try:
                out = self.analyse(f["path"])
            except Exception as e:
                errors.append(str(e)); out = {}
            return {"timestamp": f["timestamp"], "path": f["path"],
                    "frame_type": out.get("frame_type", "unknown"),
                    "ocr_text": (out.get("ocr_text") or "").strip(),
                    "description": (out.get("description") or "").strip()}
        with ThreadPoolExecutor(max_workers=3) as ex:
            frames = list(ex.map(work, kept))
        if errors: print("   Vision warning (first error):", errors[0][:300])
        self.bb.write("frames", frames)
        n_text = sum(1 for f in frames if f["ocr_text"])
        self.say("Fusion", f"vision ready: {len(frames)} frames, {n_text} with on-screen text")
        return frames

## Cell 8 — Audio Agent (emphasis & quiet moments with librosa)

In [9]:
# CELL 8 - Audio Agent (speaker diarization with pyannote needs a HuggingFace token, so we do loudness analysis instead)
import librosa

class AudioAgent(Agent):
    name = "Audio"
    def run(self, media, win=5.0):
        y, sr = librosa.load(media["audio"], sr=16000, mono=True)
        hop = int(win * sr); n = len(y) // hop
        if n < 4: return []
        rms = np.array([np.sqrt(np.mean(y[i*hop:(i+1)*hop] ** 2) + 1e-12) for i in range(n)])
        hi, med = np.percentile(rms, 92), np.median(rms)
        events = []
        for i, v in enumerate(rms):
            if v >= hi: events.append({"start": i * win, "end": (i + 1) * win, "kind": "emphasis"})
            elif v < 0.25 * med: events.append({"start": i * win, "end": (i + 1) * win, "kind": "quiet"})
        self.bb.write("audio_events", events)
        self.say("Fusion", f"audio ready: {sum(e['kind']=='emphasis' for e in events)} emphasised windows")
        return events

## Cell 9 — Fusion Agent + Segmentation Agent (one timeline, topic-aware chunks)

In [26]:
# CELL 9 - Fusion & Segmentation
from sentence_transformers import SentenceTransformer
EMB = SentenceTransformer("all-MiniLM-L6-v2")
def embed(texts):
    return np.asarray(EMB.encode(list(texts), normalize_embeddings=True, show_progress_bar=False))

OCR_TYPES = {"slide", "code_editor", "chart", "whiteboard", "demo"}

class FusionAgent(Agent):
    name = "Fusion"
    def run(self, transcript, frames, audio_events):
        fr = sorted(frames, key=lambda f: f["timestamp"])
        emph = [e for e in audio_events if e["kind"] == "emphasis"]
        timeline, j = [], -1
        for s in transcript["segments"]:
            while j + 1 < len(fr) and fr[j + 1]["timestamp"] <= s["start"] + 1: j += 1
            timeline.append({"start": s["start"], "end": s["end"], "text": s["text"],
                             "frame_idx": j if j >= 0 else None,
                             "emphasis": any(e["start"] < s["end"] and e["end"] > s["start"] for e in emph)})
        self.bb.write("frames", fr); self.bb.write("timeline", timeline)
        self.say("Segmenter", f"unified timeline: {len(timeline)} segments aligned with {len(fr)} frames")
        return timeline

class SegmentationAgent(Agent):
    name = "Segmenter"

    def make_blocks(self, timeline, target_words=70):
        blocks, cur = [], None
        for e in timeline:
            if cur is None: cur = {"start": e["start"], "end": e["end"], "text": e["text"], "emph": e["emphasis"]}
            else: cur["text"] += " " + e["text"]; cur["end"] = e["end"]; cur["emph"] = cur["emph"] or e["emphasis"]
            if len(cur["text"].split()) >= target_words: blocks.append(cur); cur = None
        if cur: blocks.append(cur)
        return blocks

    def screen_lines(self, frames, start, end):
        prior = [f for f in frames if f["timestamp"] <= start]
        inside = [f for f in frames if start < f["timestamp"] <= end]
        lines = []
        for f in (prior[-1:] + inside):
            if f["frame_type"] in OCR_TYPES and (f["ocr_text"] or f["description"]):
                lines.append(f"[{fmt_ts(f['timestamp'])}] ({f['frame_type']}) OCR: {f['ocr_text'][:400]} | {f['description'][:200]}")
        return "\n".join(lines[:8])

    def run(self, timeline, frames):
        blocks = self.make_blocks(timeline)
        E = embed([b["text"] for b in blocks])
        k, sims = 3, []
        for i in range(1, len(blocks)):
            a, b = E[max(0, i - k):i].mean(0), E[i:i + k].mean(0)
            sims.append(float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)))
        sims = np.array(sims)
        bounds = set()
        if len(sims) > 2:
            thr = sims.mean() - 0.5 * sims.std()
            for i, s in enumerate(sims):
                left_ok = (i == 0) or s <= sims[i - 1]
                right_ok = (i == len(sims) - 1) or s <= sims[i + 1]
                if s < thr and left_ok and right_ok: bounds.add(i + 1)
        groups, cur, words = [], [], 0
        for i, b in enumerate(blocks):
            w = len(b["text"].split())
            if cur and ((i in bounds and words >= CFG["MIN_CHUNK_WORDS"]) or words + w > CFG["CHUNK_WORDS"]):
                groups.append(cur); cur, words = [], 0
            cur.append(b); words += w
        if cur: groups.append(cur)

        # FIX: merge a tiny trailing group safely (the old one-liner crashed on a 2-element list)
        if len(groups) > 1:
            last_words = sum(len(b["text"].split()) for b in groups[-1])
            if last_words < CFG["MIN_CHUNK_WORDS"] // 2:
                last = groups.pop()
                groups[-1] += last

        chunks = []
        for i, g in enumerate(groups):
            s, e = g[0]["start"], g[-1]["end"]
            chunks.append({"id": i, "start": s, "end": e,
                           "text": "\n".join(f"[{fmt_ts(b['start'])}] {b['text']}" for b in g),
                           "screen_text": self.screen_lines(frames, s, e),
                           "words": sum(len(b["text"].split()) for b in g),
                           "emphasis": any(b["emph"] for b in g)})
        self.bb.write("blocks", blocks); self.bb.write("block_emb", E); self.bb.write("chunks", chunks)
        self.say("Summarizer", f"{len(chunks)} topic-aware chunks ({sum(c['words'] for c in chunks)} words)")
        return chunks

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Cell 10 — Summarizer Agent (map → chapters → reduce)

In [11]:
# CELL 10 - Summarizer Agent (hierarchical map-reduce)
SYS_NOTES = ("You are a precise note-taker. Use ONLY the supplied material. Never add outside facts. "
             "If you are unsure, leave it out.")

class SummarizerAgent(Agent):
    name = "Summarizer"

    def lang_rule(self, out_lang):
        return "" if out_lang.lower().startswith("en") else f" Write all string values in {out_lang} (keep JSON keys in English)."

    def map_chunk(self, ch, out_lang):
        prompt = ("TRANSCRIPT CHUNK (timestamps in [mm:ss]):\n" + ch["text"] +
                  "\n\nON-SCREEN CONTENT (OCR/vision):\n" + (ch["screen_text"] or "none") +
                  '\n\nReturn JSON: {"title": "short chapter title", "summary": "3-5 sentences", '
                  '"key_points": [{"point": "...", "ts": "mm:ss copied from the nearest timestamp marker"}], '
                  '"on_screen": ["important slide titles, formulas or code, if any"]}. Give 3-5 key points.'
                  + self.lang_rule(out_lang))
        try:
            out = llm_json(prompt, system=SYS_NOTES, max_tokens=900)
        except Exception as e:
            print("   map error:", str(e)[:150]); out = {}
        kps = []
        for kp in (out.get("key_points") or []):
            if isinstance(kp, dict) and kp.get("point"):
                kps.append({"point": str(kp["point"]), "ts": parse_ts(kp.get("ts")) if parse_ts(kp.get("ts")) is not None else ch["start"]})
        return {"id": ch["id"], "start": ch["start"], "end": ch["end"],
                "title": out.get("title") or f"Part {ch['id'] + 1}",
                "summary": out.get("summary") or ch["text"][:300],
                "key_points": kps, "on_screen": [str(x) for x in (out.get("on_screen") or [])][:6]}

    def map_all(self, chunks, out_lang):
        self.say("Orchestrator", f"MAP: summarising {len(chunks)} chunks with {CFG['LLM_WORKERS']} workers")
        with ThreadPoolExecutor(max_workers=CFG["LLM_WORKERS"]) as ex:
            return list(ex.map(lambda c: self.map_chunk(c, out_lang), chunks))

    def make_chapters(self, notes, out_lang):
        n = len(notes)
        groups, titles = [(i, i) for i in range(n)], {}
        if n > 12:
            listing = "\n".join(f"{i}: [{fmt_ts(x['start'])}] {x['title']} - {x['summary'][:160]}" for i, x in enumerate(notes))
            out = {}
            try:
                out = llm_json('Group these consecutive video parts into 6-12 chapters. Return JSON {"chapters":[{"title":"...","from_id":int,"to_id":int}]} '
                               f"covering ids 0..{n-1} in order with no gaps.\n" + listing + self.lang_rule(out_lang), max_tokens=800)
            except Exception as e:
                print("   chapter grouping error:", str(e)[:120])
            chs, nxt, valid, g2, t2 = out.get("chapters") or [], 0, True, [], {}
            for c in chs:
                try: a, b = int(c["from_id"]), int(c["to_id"])
                except Exception: valid = False; break
                if a != nxt or b < a or b >= n: valid = False; break
                g2.append((a, b)); t2[(a, b)] = c.get("title"); nxt = b + 1
            if valid and nxt == n and g2: groups, titles = g2, t2
            else:
                size = math.ceil(n / 10); groups = [(i, min(i + size - 1, n - 1)) for i in range(0, n, size)]
        chapters = []
        for a, b in groups:
            sub = notes[a:b + 1]
            chapters.append({"title": titles.get((a, b)) or sub[0]["title"], "start": sub[0]["start"], "end": sub[-1]["end"],
                             "summary": " ".join(x["summary"] for x in sub),
                             "key_points": [kp for x in sub for kp in x["key_points"]][:6],
                             "on_screen": list(dict.fromkeys(s for x in sub for s in x["on_screen"]))[:8]})
        return chapters

    def reduce(self, chapters, meta, out_lang):
        self.say("Orchestrator", "REDUCE: building final summary, takeaways and key moments")
        listing = "\n\n".join(
            f"CHAPTER [{fmt_ts(c['start'])}] {c['title']}\n{c['summary']}\nKey points: " +
            " | ".join(f"({fmt_ts(k['ts'])}) {k['point']}" for k in c["key_points"][:4]) for c in chapters)
        prompt = (f"Video title: {meta['title']}\n\n{listing}\n\n"
                  'Return JSON: {"tldr": "ONE sentence", "bullets": ["exactly 5 bullets summarising the whole video"], '
                  '"takeaways": [{"text": "...", "ts": "mm:ss copied from a key point"}], '
                  '"key_moments": [{"ts": "mm:ss", "why": "..."}]}. '
                  "Give 6-10 takeaways and the 5 most important key moments. Use only the notes above." + self.lang_rule(out_lang))
        out = llm_json(prompt, system=SYS_NOTES, max_tokens=1500)
        def tsn(x, default): v = parse_ts(x); return v if v is not None else default
        return {"tldr": str(out.get("tldr") or ""),
                "bullets": [str(b) for b in (out.get("bullets") or [])][:6],
                "takeaways": [{"text": str(t.get("text", "")), "ts": tsn(t.get("ts"), chapters[0]["start"])}
                              for t in (out.get("takeaways") or []) if isinstance(t, dict) and t.get("text")],
                "key_moments": [{"ts": tsn(t.get("ts"), chapters[0]["start"]), "why": str(t.get("why", ""))}
                                for t in (out.get("key_moments") or []) if isinstance(t, dict)]}

    def run(self, chunks, meta, out_lang="English"):
        notes = self.map_all(chunks, out_lang)
        chapters = self.make_chapters(notes, out_lang)
        summary = self.reduce(chapters, meta, out_lang)
        self.bb.write("notes", notes)
        return chapters, summary

## Cell 11 — Critic Agent (fact-check with retrieval + feedback loop)

In [12]:
# CELL 11 - Critic Agent
class CriticAgent(Agent):
    name = "Critic"

    def _frame_at(self, frames, t):
        best = None
        for f in frames:
            if f["timestamp"] <= t + 1: best = f
            else: break
        return best

    def evidence(self, text, ts=None, span=None, k=3):
        blocks, E, frames = self.bb.read("blocks"), self.bb.read("block_emb"), self.bb.read("frames", [])
        sc = E @ embed([text])[0]
        cand = [i for i, b in enumerate(blocks) if span and b["end"] >= span[0] and b["start"] <= span[1]] or list(range(len(blocks)))
        top = sorted(cand, key=lambda i: -sc[i])[:k]
        if ts is not None: top.append(min(range(len(blocks)), key=lambda i: abs(blocks[i]["start"] - ts)))
        top = sorted(set(top)); lines, seen = [], set()
        for i in top:
            b = blocks[i]; lines.append(f"[{fmt_ts(b['start'])}] {b['text']}")
            f = self._frame_at(frames, b["start"])
            if f and f["frame_type"] in OCR_TYPES and f["ocr_text"] and f["timestamp"] not in seen:
                seen.add(f["timestamp"]); lines.append(f"[ON-SCREEN {fmt_ts(f['timestamp'])}] {f['ocr_text'][:300]} {f['description'][:150]}")
        return "\n".join(lines)

    def collect(self, summary, chapters):
        claims = []
        def add(kind, idx, text, ts=None, span=None): claims.append({"cid": len(claims), "kind": kind, "idx": idx, "text": text, "ts": ts, "span": span})
        if summary["tldr"]: add("tldr", 0, summary["tldr"])
        for i, b in enumerate(summary["bullets"]): add("bullet", i, b)
        for i, t in enumerate(summary["takeaways"]): add("takeaway", i, t["text"], ts=t["ts"])
        for i, c in enumerate(chapters): add("chapter", i, c["summary"], span=(c["start"], c["end"]))
        return claims

    def verify(self, claims):
        results = {}
        for i in range(0, len(claims), 5):
            batch, parts = claims[i:i + 5], []
            for c in batch:
                parts.append(f"CLAIM {c['cid']}: {c['text']}\nEVIDENCE:\n{self.evidence(c['text'], c['ts'], c['span'])}")
            prompt = ("You are a strict fact-checker. For each claim decide whether the EVIDENCE (transcript plus on-screen text) "
                      "supports every factual statement in it. If it is not fully supported, give corrected_text: a corrected version "
                      'using ONLY the evidence, or "" if the claim cannot be salvaged.\n\n' + "\n\n".join(parts) +
                      '\n\nReturn JSON: {"verdicts": [{"id": <claim number>, "supported": true or false, "reason": "short", "corrected_text": ""}]}')
            try:
                out = llm_json(prompt, max_tokens=1400)
            except Exception as e:
                print("   critic error:", str(e)[:120]); out = {}
            for v in out.get("verdicts") or []:
                try: results[int(v["id"])] = v
                except Exception: pass
        return results

    def apply(self, c, summary, chapters):
        if c["kind"] == "tldr": summary["tldr"] = c["text"]
        elif c["kind"] == "bullet": summary["bullets"][c["idx"]] = c["text"]
        elif c["kind"] == "takeaway": summary["takeaways"][c["idx"]]["text"] = c["text"]
        elif c["kind"] == "chapter": chapters[c["idx"]]["summary"] = c["text"]

    def run(self, summary, chapters):
        claims = self.collect(summary, chapters); total = len(claims)
        self.say("Summarizer", f"draft v1 received: verifying {total} claims against the transcript")
        res = self.verify(claims)
        bad = [c for c in claims if not res.get(c["cid"], {}).get("supported", True)]
        init_faith = (total - len(bad)) / max(total, 1)
        revised = []
        for c in bad:
            v = res[c["cid"]]; new = (v.get("corrected_text") or "").strip()
            self.say("Summarizer", f"claim {c['cid']} ({c['kind']}) not fully supported: {str(v.get('reason', ''))[:90]}")
            if new: c["text"] = new; self.apply(c, summary, chapters); revised.append(c)
            else: c["drop"] = True
        if revised:
            self.say("Summarizer", f"draft v2 ready ({len(revised)} claims corrected) -> re-verify")
            res2 = self.verify(revised)
            for c in revised:
                if not res2.get(c["cid"], {}).get("supported", True): c["drop"] = True
        dropped = {(c["kind"], c["idx"]) for c in claims if c.get("drop")}
        summary["bullets"] = [b for i, b in enumerate(summary["bullets"]) if ("bullet", i) not in dropped]
        summary["takeaways"] = [t for i, t in enumerate(summary["takeaways"]) if ("takeaway", i) not in dropped]
        if ("tldr", 0) in dropped: summary["tldr"] = ""
        for i, c in enumerate(chapters):
            if ("chapter", i) in dropped:
                c["summary"] = "(Removed: could not be verified against the transcript.)"; c["unverified"] = True
        report = {"claims_checked": total, "initial_faithfulness": round(init_faith, 3),
                  "corrected": len(revised), "removed": len(dropped)}
        self.say("Orchestrator", f"approved. initial faithfulness {init_faith:.0%}, corrected {len(revised)}, removed {len(dropped)}")
        self.bb.write("critic_report", report)
        return summary, chapters, report

## Cell 12 — Q&A Agent (RAG over the timeline, with timestamp citations)

In [13]:
# CELL 12 - Q&A Agent
class QAAgent(Agent):
    name = "Q&A"
    def __init__(self, bb):
        super().__init__(bb)
        self.blocks, self.E, self.frames, self.history = bb.read("blocks"), bb.read("block_emb"), bb.read("frames", []), []

    def active_frame(self, t):
        best = None
        for f in self.frames:
            if f["timestamp"] <= t + 1: best = f
            else: break
        return best

    def ask(self, question, k=5):
        sc = self.E @ embed([question])[0]
        idx = sorted(np.argsort(-sc)[:k].tolist())
        ctx, seen = [], set()
        for i in idx:
            b = self.blocks[i]; ctx.append(f"[{fmt_ts(b['start'])}] {b['text']}")
            f = self.active_frame(b["start"])
            if f and f["frame_type"] in OCR_TYPES and f["ocr_text"] and f["timestamp"] not in seen:
                seen.add(f["timestamp"]); ctx.append(f"(on screen at {fmt_ts(f['timestamp'])}: {f['ocr_text'][:300]})")
        system = ("You answer questions about ONE video using ONLY the context excerpts. After each claim cite the timestamp in the "
                  "form [mm:ss], copied from the excerpt headers. If the answer is not in the excerpts, say the video does not seem to cover it.")
        msgs = [{"role": "system", "content": system}] + self.history[-6:] + \
               [{"role": "user", "content": "CONTEXT:\n" + "\n\n".join(ctx) + "\n\nQUESTION: " + question}]
        ans = llm(msgs, max_tokens=700)
        self.history += [{"role": "user", "content": question}, {"role": "assistant", "content": ans}]
        return ans

## Cell 13 — Orchestrator (plans the job, runs agents in parallel, handles failures)

In [27]:
# CELL 13 - Orchestrator
class Orchestrator(Agent):
    name = "Orchestrator"
    def __init__(self, bb=None): super().__init__(bb or Blackboard())

    def plan(self, meta):
        table = {"slide_lecture": (CFG["MAX_VISION_FRAMES"], 8), "code_tutorial": (CFG["MAX_VISION_FRAMES"], 8),
                 "talk_demo": (min(18, CFG["MAX_VISION_FRAMES"]), 10), "podcast_interview": (6, 30), "other": (12, 12)}
        vt = "other"
        try:
            out = llm_json('Classify this video from its metadata. Return JSON {"video_type": "slide_lecture" | "code_tutorial" | "talk_demo" | "podcast_interview" | "other", "reason": "short"}\n'
                           f"Title: {meta['title']}\nChannel: {meta.get('channel', '')}\nDescription: {meta.get('description', '')[:600]}",
                           model=CFG["FAST_MODEL"], max_tokens=150)
            vt = out.get("video_type", "other")
        except Exception as e:
            print("   plan fallback:", str(e)[:100])
        if vt not in table: vt = "other"
        budget, every = table[vt]
        self.say("All", f"video type = {vt}. Plan: vision budget {budget} frames (sample every {every}s) + ASR + audio analysis")
        return {"video_type": vt, "vision_budget": budget, "sample_every": every}

    def run(self, url=None, local_file=None, cookies_file=None, lang_hint=None, out_lang="English",
            force_whisper=False, use_vision=True, use_audio=True):
        t0 = time.time()
        bb = self.bb = Blackboard()
        media = IngestionAgent(bb).run(url, local_file, cookies_file)
        meta = media["meta"]
        plan = self.plan(meta)
        budget = plan["vision_budget"] if use_vision else 0

        with ThreadPoolExecutor(max_workers=3) as ex:
            f_t = ex.submit(TranscriptAgent(bb).run, media, lang_hint, force_whisper)
            f_v = ex.submit(VisionAgent(bb).run, media, budget, plan["sample_every"]) if budget > 0 else None
            f_a = ex.submit(AudioAgent(bb).run, media) if use_audio else None
            transcript = f_t.result()                      # transcript is mandatory
            frames, events = [], []
            if f_v:
                try: frames = f_v.result()
                except Exception as e: print("Vision agent failed - continuing without vision:", str(e)[:200])
            if f_a:
                try: events = f_a.result()
                except Exception as e: print("Audio agent failed - continuing without it:", str(e)[:200])

        timeline = FusionAgent(bb).run(transcript, frames, events)
        chunks = SegmentationAgent(bb).run(timeline, bb.read("frames"))
        chapters, summary = SummarizerAgent(bb).run(chunks, meta, out_lang)
        summary, chapters, report = CriticAgent(bb).run(summary, chapters)
        bb.write("qa", QAAgent(bb))
        self.say("Q&A", "timeline indexed for chat")

        elapsed = time.time() - t0
        result = {"meta": meta, "plan": plan, "transcript_source": transcript["source"], "language": transcript["language"],
                  "frames": bb.read("frames"), "chunks": chunks, "chapters": chapters, "summary": summary,
                  "critic": report, "elapsed_sec": elapsed}
        bb.write("result", result)
        ratio = elapsed / max(meta["process_seconds"], 1)
        print(f"\nDone in {elapsed/60:.1f} min ({ratio:.0%} of processed duration).")
        return result

ORCH = Orchestrator()
print("Orchestrator ready.")

Orchestrator ready.


## Cell 14 — Rendering helpers (summary → Markdown with clickable timestamps)

In [15]:
# CELL 14 - Rendering
from IPython.display import display, Markdown, HTML

def ts_link(meta, sec):
    label = fmt_ts(sec)
    if meta.get("url") and not str(meta.get("id", "")).startswith("local_"):
        return f"[{label}](https://youtu.be/{meta['id']}?t={int(sec)})"
    return f"`{label}`"

def linkify(text, meta):
    return re.sub(r"\[(\d{1,2}:\d{2}(?::\d{2})?)\]", lambda m: ts_link(meta, parse_ts(m.group(1))), text)

def render_markdown(res, level="detailed"):
    meta, s, ch = res["meta"], res["summary"], res["chapters"]
    L = [f"# {meta['title']}",
         f"*{meta.get('channel', '')} | length {fmt_ts(meta['duration'])} | processed first {fmt_ts(meta['process_seconds'])} | "
         f"transcript: {res['transcript_source']} | video type: {res['plan']['video_type']}*"]
    if s["tldr"]: L.append(f"\n**TL;DR:** {s['tldr']}")
    if level in ("bullets", "detailed") and s["bullets"]:
        L.append("\n## Summary\n" + "\n".join(f"- {b}" for b in s["bullets"]))
    L.append("\n## Chapters\n" + "\n".join(f"- {ts_link(meta, c['start'])} **{c['title']}**" for c in ch))
    if level in ("bullets", "detailed"):
        if s["takeaways"]:
            L.append("\n## Key takeaways\n" + "\n".join(f"- {t['text']} ({ts_link(meta, t['ts'])})" for t in s["takeaways"]))
        if s["key_moments"]:
            L.append("\n## Key moments (watch these first)\n" + "\n".join(f"- {ts_link(meta, k['ts'])} - {k['why']}" for k in s["key_moments"]))
    if level == "detailed":
        L.append("\n## Section-by-section report")
        for c in ch:
            L.append(f"\n### {ts_link(meta, c['start'])} {c['title']}\n{c['summary']}")
            if c["key_points"]:
                L.append("\n" + "\n".join(f"- {k['point']} ({ts_link(meta, k['ts'])})" for k in c["key_points"]))
        shown = [f for f in res["frames"] if f["ocr_text"] and f["frame_type"] in OCR_TYPES][:25]
        if shown:
            L.append("\n## On-screen content (OCR)")
            for f in shown:
                txt = f["ocr_text"][:500].replace("\n", "  \n> ")
                L.append(f"\n**{ts_link(meta, f['timestamp'])}** ({f['frame_type']})\n> {txt}")
                if f["description"]: L.append(f"\n*Visual:* {f['description']}")
    c = res["critic"]
    L.append(f"\n---\n*Critic: {c['claims_checked']} claims checked, initial faithfulness {c['initial_faithfulness']:.0%}, "
             f"{c['corrected']} corrected, {c['removed']} removed. Processing time {res['elapsed_sec']/60:.1f} min.*")
    return "\n".join(L)

## Cell 15 — (Optional) Upload a video file or cookies.txt

Use this if the YouTube download is blocked on Colab. Download the video on your own computer, then upload it here. **Skip this cell if the normal URL works.**

In [16]:
# CELL 15 (OPTIONAL) - upload your own video / cookies.txt
from google.colab import files
LOCAL_FILE, COOKIES_FILE = None, None

print("Upload a VIDEO file (mp4/mkv/webm), or cancel to skip:")
up = files.upload()
if up: LOCAL_FILE = "/content/" + list(up.keys())[0]; print("LOCAL_FILE =", LOCAL_FILE)

# To use cookies instead, uncomment:
# print("Upload cookies.txt (Netscape format):")
# up = files.upload()
# if up: COOKIES_FILE = "/content/" + list(up.keys())[0]; print("COOKIES_FILE =", COOKIES_FILE)

Upload a VIDEO file (mp4/mkv/webm), or cancel to skip:


Saving Slides Presentation.mp4 to Slides Presentation.mp4
LOCAL_FILE = /content/Slides Presentation.mp4


## Cell 16 — ▶️ RUN ClipSense

Edit the settings, then run. Watch the agents talk to each other in the log. Expect roughly 3-8 minutes for a 30-45 minute video (free-tier rate limits cause short waits).

In [28]:
# CELL 16 - RUN
URL = "https://www.youtube.com/watch?v=aircAruvnKk"   # <- paste any YouTube link (this one is a ~19 min neural-network explainer)
try: LOCAL_FILE
except NameError: LOCAL_FILE = None
try: COOKIES_FILE
except NameError: COOKIES_FILE = None

OUTPUT_LANGUAGE = "English"     # e.g. "Urdu", "Hindi", "Spanish"
SUMMARY_LEVEL   = "detailed"    # "tldr" | "bullets" | "detailed"
FORCE_WHISPER   = False         # True = ignore YouTube captions and use Whisper
USE_VISION, USE_AUDIO = True, True
# CFG["MAX_MINUTES"] = 45       # raise/lower how much of the video is processed

RESULT = ORCH.run(url=None if LOCAL_FILE else URL, local_file=LOCAL_FILE, cookies_file=COOKIES_FILE,
                  out_lang=OUTPUT_LANGUAGE, force_whisper=FORCE_WHISPER, use_vision=USE_VISION, use_audio=USE_AUDIO)
display(Markdown(render_markdown(RESULT, SUMMARY_LEVEL)))

[    0.2s]    Ingestion -> Orchestrator ready: 'Slides Presentation.mp4' (06:55), processing first 06:55
[    0.6s] Orchestrator -> All          video type = slide_lecture. Plan: vision budget 25 frames (sample every 8s) + ASR + audio analysis
[    0.9s]        Audio -> Fusion       audio ready: 7 emphasised windows
   whisper chunk 1/1 done
[    2.6s]   Transcript -> Fusion       transcript ready: 30 segments, 959 words (whisper)
[   28.7s]       Vision -> Orchestrator 21 visually-unique frames after dedupe
[   28.7s]       Vision -> Orchestrator analysing 21 frames with qwen3.8-27b
   [vision] rate limit - waiting 20s (retry 1/6)
   [vision] rate limit - waiting 19s (retry 1/6)
   [vision] rate limit - waiting 19s (retry 1/6)
   [vision] rate limit - waiting 18s (retry 2/6)
   [vision] rate limit - waiting 18s (retry 2/6)
   [vision] rate limit - waiting 19s (retry 1/6)
   [vision] rate limit - waiting 45s (retry 3/6)
   [vision] rate limit - waiting 45s (retry 3/6)
   [vision] rate 

# Slides Presentation.mp4
* | length 06:55 | processed first 06:55 | transcript: whisper | video type: slide_lecture*

**TL;DR:** An AI-driven system monitors construction sites in real time, detecting safety hazards and alerting officers via a dashboard and email.

## Summary
- Continuous video monitoring addresses the limitation of human supervisors who cannot watch all cameras.
- The pipeline normalizes video with FFmpeg and runs parallel YOLOv8 models for PPE, fire/smoke, fights, fallen workers, worker tracking, and structural anomalies.
- PPE compliance is visualized with green (compliant) and red (violation) bounding boxes per worker.
- Alerts are aggregated, include cooldown mechanisms, and are sent as annotated emails to prevent spamming.
- A web‑based dashboard displays detections, headcounts, confidence scores, and supports site safety officers for faster response.

## Chapters
- `00:00` **AI-Based Construction Site Surveillance Overview**

## Key takeaways
- Human supervisors cannot monitor all camera feeds, leading to missed accidents. (`00:32`)
- Video is normalized using FFmpeg before AI processing. (`01:14`)
- PPE compliance model classifies helmets and vests, showing green boxes for compliance and red for violations. (`02:06`)
- A heuristic structural anomaly detector adds another safety layer. (`01:14`)
- The system sends annotated email alerts with a cooldown mechanism and provides a dashboard that visualizes detections. (`00:00`)

## Key moments (watch these first)
- `00:32` - Highlights the need for continuous monitoring because human supervisors miss incidents.
- `01:14` - Explains the AI detection pipeline, including video normalization and parallel YOLOv8 models.
- `02:06` - Demonstrates PPE compliance detection with visual green/red bounding boxes.
- `02:42` - Shows altercation and fallen‑worker detection with cooldown to avoid alert spamming.
- `00:00` - Provides the overall solution overview targeting safety officers and improving response times.

## Section-by-section report

### `00:00` AI-Based Construction Site Surveillance Overview
The team presents an AI-driven system that continuously monitors construction sites to detect safety hazards such as PPE violations, fire, smoke, fights, and fallen workers. Video input is normalized with FFmpeg and processed through multiple YOLOv8 models and a heuristic structural anomaly detector, producing bounding boxes, labels, and alerts. A web‑based dashboard visualizes detections, headcounts, and confidence scores, while an alerting module sends annotated emails with cooldown to avoid spamming. The solution targets site safety officers and aims to improve response times, reliability, and scalability.

- Continuous monitoring is needed because human supervisors cannot watch all cameras at all times, leading to missed accidents. (`00:32`)
- The AI detection pipeline includes video normalization, parallel safety condition models (PPE, fire/smoke, fight, fallen worker, worker tracking, structural anomaly), and aggregation into alerts. (`01:14`)
- PPE compliance uses a construction‑PPE YOLOv8 model to classify helmets and vests per worker, showing green boxes for compliance and red for violations. (`02:06`)
- Altercation and fallen‑worker detection rely on community‑trained YOLOv8 models, with cooldown mechanisms to limit repeated alerts. (`02:42`)
- The web dashboard displays bounding boxes, PPE status, worker IDs, headcounts, and sends email alerts with annotated screenshots when critical events occur. (`04:07`)

## On-screen content (OCR)

**`00:00`** (slide)
> PAKANGELS  
> GENERATIVE & AGENTIC AI  
> PROJECT PRESENTATION · APPLIED COMPUTER VISION FOR INDUSTRIAL SAFETY  
> AI-Based Construction Site Surveillance System  
> An End-to-End Multi-Model Computer Vision System for Real-Time Safety Monitoring, Predictive Risk Analysis and Automated Emergency Alerting  
> PRESENTED BY · TEAM OF 5  
> Muhammad Talha (Team Lead) · Muhammad Haseen · Gule Zahra Alvi · Mahreen Sajjad · Tooba Rafique  
> PakAngels · Generative & Agentic AI Program · Cohort 11

*Visual:* A presentation title slide featuring a background image of a construction crane at dusk, with the project title and a list of team members.

**`00:16`** (slide)
> PRESENTED BY  
> Meet the Team  
> Five members - PakAngels Generative & Agentic AI, Cohort 11  
> TEAM LEAD  
> Muhammad Talha  
> Project direction, pipeline architecture and integration of the detection engine with the dashboard.  
> Architecture  
> Integration  
> Direction  
> Muhammad Hasnain  
> Team Member  
> Model sourcing and inference tuning across the YOLOv8 detector family.  
> Gule Zehra Alvi  
> Team Member  
> Event logging, analytics suite and the predictive site risk scoring logic.  
> Maheen Sajjad  
> Team Member  
> Frontend dashboard desi

*Visual:* A presentation slide titled 'Meet the Team' featuring a grid of five profile cards. The left card highlights the Team Lead, Muhammad Talha, with an orange background and skill tags, while the other four cards list team members with their specific roles and responsibilities.

**`00:24`** (slide)
> PRESENTED BY  
> Meet the Team  
> Five members - PakAngels Generative & Agentic AI, Cohort 11  
> TEAM LEAD  
> Muhammad Talha  
> Project direction, pipeline architecture and integration of the detection engine with the dashboard.  
> Architecture  
> Integration  
> Direction  
> Muhammad Hasnain  
> Team Member  
> Model sourcing and inference tuning across the YOLOv8 detector family.  
> Gule Zehra Alvi  
> Team Member  
> Event logging, analytics suite and the predictive site risk scoring logic.  
> Maheen Sajjad  
> Team Member  
> Frontend dashboard desi

*Visual:* A presentation slide titled 'Meet the Team' featuring a large orange profile card for the team lead and four smaller white cards for team members, each with an icon, name, role, and a brief description of their responsibilities.

**`00:32`** (slide)
> SECTION 01 · THE PROBLEM  
> Hazards Are Visible — But Nobody Is Watching  
> Construction is one of the most hazardous industries in the world. U.S. Bureau of Labor Statistics data shows construction workers account for roughly one in five workplace fatalities nationally, at a fatality rate several times the all-industry average.  
> OSHA'S "FATAL FOUR"  
> Falls  
> Struck-by  
> Electrocution  
> Caught-in/between  
> Together these account for the large majority of construction deaths each year — falls alone remain the sin

*Visual:* A presentation slide featuring a donut chart showing construction's 20% share of U.S. workplace fatalities, a section listing OSHA's 'Fatal Four' hazards, and a red callout box highlighting the root cause of monitoring failures.

**`00:56`** (slide)
> SECTION 02 - OUR SOLUTION  
> Continuous Monitoring That Never Looks Away  
> An end-to-end, multi-model computer vision system that continuously analyses construction site footage, flags safety violations and emergencies in real time, alerts supervisors instantly, and surfaces predictive risk before incidents escalate.  
> Part 1 — AI Detection Pipeline  
> GPU-accelerated inference engine, developed and benchmarked on Kaggle (NVIDIA T4 / P100).  
> Part 2 — Web Monitoring Dashboard  
> Built in Visual Studio Code, tu

*Visual:* A presentation slide detailing a computer vision solution with two main components (AI pipeline and dashboard), a photo of security cameras, and a row of metric cards.

**`01:28`** (slide)
> How a Frame Becomes an Alert  
> Construction-Site AI Surveillance — Processing Pipeline  
> INPUT  
> (Image / Video)  
> Frame extraction  
> (ffmpeg normalised)  
> PPE Model  
> (Helmet + Vest)  
> Fire & Smoke  
> Model  
> Fight / Violence  
> Model  
> Fallen / Stalker  
> Model  
> Safe/Back  
> Worker  
> Classifier  
> Structured JSON  
> (bounding boxes, labels)  
> Aggregation & Overlay Engine  
> (boxes, alert banners, worker-count HUD)  
> Annotated output  
> (image / H.264 video)  
> Event log (DataFrames)  
> → charts & dashboards  
> Figure 1 — System processing pipeline, gen

*Visual:* A flowchart diagram illustrating a computer vision processing pipeline, showing data flow from input through parallel detection models to aggregation and final outputs, accompanied by a numbered list of steps on the right.

**`02:08`** (slide)
> PPE Compliance and Fire & Smoke Detection  
> MODEL - HUGGING FACE  
> kiliuminati1 / construction-ppe-yolov8  
> YOLOv8 - 19 classes - mAP50 = 0.76  
> HOW IT WORKS  
> Per-worker classification  
> Detects each person in frame, then classifies helmet and vest presence individually  
> Green box, red box, event log  
> Compliant workers are boxed green and violations red, each written to the timestamped log  
> KEY BENEFITS  
> Every worker checked, every frame  
> Replaces occasional manual spot-checks with continuous, unbiased inspecti

*Visual:* A presentation slide detailing two AI models for site surveillance: one for PPE compliance using YOLOv8 and another for fire and smoke detection, including images of construction workers and a fire, along with sections on how they work and their benefits.

**`02:40`** (slide)
> CAPABILITIES 03 - 04  
> Altercation and Fallen-Worker Detection  
> MODEL - HUGGING FACE  
> Musawer14 / fight_detection_yolov8  
> YOLOv8-nano - community trained - no published benchmark  
> HOW IT WORKS  
> Fight vs. normal activity  
> Trained on a labelled fight / no-fight surveillance dataset, on the same frame stream.  
> Critical event, with cooldown  
> Raises the alarm banner and emails the owner; one incident yields one notification.  
> WHERE ELSE THIS APPLIES  
> Schools & hostels - University campuses - Metro & transit stat

*Visual:* A presentation slide detailing two AI detection models for altercation and fallen-worker scenarios, featuring screenshots of workers and text sections explaining how the models work and their key benefits.

**`03:20`** (slide)
> CAPABILITIES 05 - 06  
> Live Headcount and Structural Anomaly  
> TRACKING - BYTETRACK  
> ByteTrack worker counter  
> Live in-frame count - unique individual tracking  
> HOW IT WORKS  
> Live headcount HUD  
> Counts every worker in the current frame and renders the count onto the output video.  
> Persistent per-person ID  
> ByteTrack maintains an ID per individual, so the same worker is never double counted.  
> KEY BENEFITS  
> Instant evacuation headcount  
> Supervisors know how many people were in the zone and whether all are accou

*Visual:* A presentation slide detailing two surveillance capabilities: a ByteTrack worker counter and a frame-to-frame SSIM tripwire, each with sections for how it works, key benefits, and application areas, accompanied by small illustrative photos of construction workers and scaffolding.

**`04:00`** (slide)
> SECTION 06 - THE PRODUCT  
> AI Surveillance — “See. Detect. Respond.”  
> NAVIGATION — THREE FUNCTIONAL GROUPS  
> Monitor  
> Dashboard · Live Monitoring · Upload & Test · Detection Events · Alerts  
> Insights  
> Workers · PPE Compliance · Reports  
> System  
> Settings and AI service connection status  
> Live “AI service: Connected” indicator builds operator trust in system state.  
> Figure 2 — The AI Detection Lab (Upload & Test) screen of the actual application, showing a fire/smoke test case with live confidence scores.  
> Upl

*Visual:* A presentation slide featuring a screenshot of an 'AI Detection Lab' software interface on the left and a list of three navigation groups (Monitor, Insights, System) on the right.

**`04:16`** (slide)
> DETECTION OUTPUT  
> What the Operator Actually Sees  
> 01. PPE COMPLIANCE + LIVE HEADCOUNT  
> 02. FIRE & SMOKE CRITICAL ALERT  
> FIRE DETECTED  
> Green box = compliant  
> Red box = violation  
> Persistent ID + headcount  
> Full-frame banner  
> Confidence on every box  
> Smoke detected separately  
> AI-Based Construction Site Surveillance System - Pakhangeli, Agentic AI - Cohort 11

*Visual:* A presentation slide displaying two example video feeds with AI detection overlays: one showing construction workers with PPE status boxes and another showing a fire with a 'FIRE DETECTED' alert banner.

**`04:24`** (slide)
> SECTION 07 - ALERTING  
> From Detection to the Supervisor's Inbox  
> Detection  
> A detector fires on an analysed frame.  
> Classification  
> Is this informational, or critical?  
> Alarm raised  
> Pulsing banner + on-screen siren indicator.  
> Email dispatched  
> Annotated screenshot + timestamp to the site owner.  
> WHAT COUNTS AS CRITICAL  
> A single PPE violation is informational. These four event types trigger the full alarm workflow:  
> Fire & smoke  
> Fight / altercation  
> Fallen or unresponsive worker  
> Structural anomaly  
> Visual a

*Visual:* A presentation slide illustrating a four-step alerting workflow (Detection, Classification, Alarm raised, Email dispatched) and a list of critical event types and alert methods.

**`05:04`** (slide)
> SECTION 08 - PRD  
> Purpose, Objectives & Target Users  
> Product Objectives  
> To reduce preventable injuries and fatalities by providing continuous automated monitoring a human supervisor cannot physically sustain alone.  
> - Detect PPE non-compliance, fire and smoke, alterations and fallen workers in real time from standard camera footage.  
> - Notify the responsible person immediately, with enough context — screenshot, timestamp, camera ID — to act without delay.  
> - Provide a predictive, aggregated view of 

*Visual:* A presentation slide titled 'Purpose, Objectives & Target Users' featuring a blue panel listing product objectives and a column of four cards describing different user roles.

**`05:12`** (slide)
> SECTION 08 - PRD  
> Functional & Non-Functional Requirements  
> FUNCTIONAL REQUIREMENTS  
> FR-1 Detect helmet and safety-vest presence per worker in each analysed frame.  
> FR-2 Detect fire and smoke events and classify them as critical.  
> FR-3 Detect physical altercations between individuals.  
> FR-4 Detect a worker who has fallen or not moved beyond a configurable threshold.  
> FR-5 Display a live count of workers visible in the current frame.  
> FR-6 Flag sudden, large structural changes as a possible structural em

*Visual:* A presentation slide titled 'Functional & Non-Functional Requirements' displaying three columns of text: a list of functional requirements (FR-1 to FR-11), a list of non-functional requirements, and a dark blue box detailing success metrics.

**`05:36`** (slide)
> IMPACT  
> Why This Matters in the Real World  
> Faster emergency response  
> Automated detection and instant alerting removes reliance on a human watching every feed, cutting the gap between an incident and help arriving.  
> Continuous PPE compliance  
> Every worker in frame is checked consistently instead of relying on periodic manual spot-checks.  
> Early intervention via risk scoring  
> The aggregated site risk score lets supervisors act on rising risk trends before they culminate in an incident.  
> Auditable safety

*Visual:* A presentation slide titled 'Why This Matters in the Real World' displaying six benefit cards with icons and text descriptions regarding safety and compliance features.

**`06:00`** (slide)
> IMPACT  
> Why This Matters in the Real World  
> Faster emergency response  
> Automated detection and instant alerting removes reliance on a human watching every feed, cutting the gap between an incident and help arriving.  
> Continuous PPE compliance  
> Every worker in frame is checked consistently instead of relying on periodic manual spot-checks.  
> Early intervention via risk scoring  
> The aggregated site risk score lets supervisors act on rising risk trends before they culminate in an incident.  
> Auditable safety

*Visual:* A presentation slide titled 'Why This Matters in the Real World' displaying six benefit cards with icons and text descriptions arranged in a grid layout.

**`06:08`** (slide)
> ENGINEERING  
> Technology Stack  
> AI / Computer Vision  
> Python · Ultralytics YOLOv8 · OpenCV · Hugging Face Hub · PyTorch (GPU-accelerated)  
> Video Processing  
> FFmpeg — codec normalisation and H.264 encoding  
> Data & Analytics  
> Pandas · Matplotlib · Seaborn · scikit-image (SSIM)  
> Development & Experimentation  
> Kaggle Notebooks with GPU — NVIDIA T4 / P100  
> Frontend  
> Web application built in Visual Studio Code  
> Notifications  
> SMTP-based automated email alerts with image attachments  
> Built in Two Halves  
> The detection

*Visual:* A presentation slide titled 'Technology Stack' listing six engineering categories with their associated tools, alongside a blue callout box titled 'Built in Two Halves' explaining the system architecture.

**`06:32`** (slide)
> HONEST ASSESSMENT  
> Limitations & Future Scope  
> CURRENT LIMITATIONS  
> Structural detection is a heuristic  
> Rule-based SSM rather than a trained classifier, reflecting the absence of a reliable public collapse model.  
> Two models lack published benchmarks  
> Fight and fallen-worker detection are community-trained and need validation on site footage before safety-critical use.  
> Batch input only, for now  
> Uploaded image or video today — live RTSP camera ingestion is not yet wired in.  
> Dependent on camera quality

*Visual:* A presentation slide titled 'Limitations & Future Scope' displaying two columns of feature cards with icons, listing current system limitations on the left and future development goals on the right.

**`06:48`** (slide)
> PAKANGELS  
> GENERATIVE & AGENTIC AI  
> COHORT 11  
> Thank You  
> AI-Based Construction Site Surveillance System  
> We welcome your questions and feedback.  
> Muhammad Talha  
> Team Lead  
> Muhammad Hasnain  
> Team Member  
> Gule Zehra Alvi  
> Team Member  
> Maheen Sajjad  
> Team Member  
> Tooba Rafique  
> Team Member  
> PakAngels · Generative & Agentic AI Program · Cohort 11

*Visual:* A presentation closing slide with a dark blue background featuring a 'Thank You' title, a project subtitle, and a row of cards listing team members and their roles.

---
*Critic: 14 claims checked, initial faithfulness 79%, 3 corrected, 2 removed. Processing time 8.3 min.*

In [21]:
test = llm_json("Return JSON with keys: tldr, bullets", max_tokens=200)
print(test)

{'tldr': 'Brief summary of the requested information.', 'bullets': ['First key point', 'Second key point', 'Third key point']}


## Cell 17 — 💬 Chat with the video (answers cite clickable timestamps)

In [29]:
# CELL 17 - Chat
def ask(question):
    qa = ORCH.bb.read("qa")
    display(Markdown(f"**You:** {question}\n\n**ClipSense:** " + linkify(qa.ask(question), RESULT["meta"])))

ask("What are the main ideas explained in this video?")

# Interactive loop (type 'exit' to stop):
# while True:
#     q = input("Ask about the video (or 'exit'): ")
#     if q.strip().lower() in ("exit", "quit", ""): break
#     ask(q)

**You:** What are the main ideas explained in this video?

**ClipSense:** The video presents an **AI‑based construction‑site surveillance system** and explains its purpose, architecture, capabilities, and expected benefits.

* **Problem statement** – Construction sites have many serious safety hazards (falls, struck‑by incidents, etc.) and, although hazards are visible on camera, no one is constantly watching the footage. Continuous human monitoring is difficult and slow `00:00` and the need for an always‑on monitoring solution is highlighted `00:32`.

* **Overall solution** – The team proposes an end‑to‑end, multi‑model computer‑vision system that ingests video, runs AI analysis, detects hazards, and generates real‑time alerts on a web‑based dashboard `01:14` (also summarized as “continuous monitoring that never looks away”) `00:56`.

* **System architecture** – Video streams are first normalized with FFmpeg, then frames pass through several detection modules (e.g., PPE detection, worker ID and head‑count, hazard classification). The results are combined, bounding boxes and confidence scores are added, and the information is fed into an alerting workflow `01:14`.

* **Key detection capabilities** –  
  * **PPE detection** (helmet and safety‑vest presence)  
  * **Physical‑altercation detection** using a community‑trained YOLOv8 model `02:42`  
  * **Fallen‑worker detection** (classifying workers as falling, sitting, standing) `02:42`  

* **Outcome measurement and benefits** – Success is tracked with metrics such as time‑to‑alert, compliance, alert precision, and user adoption. The system promises faster safety responses, continuous PPE monitoring, scalability to more hazards, and reduced reliance on a person watching every camera `05:24`.

Together, these points outline the video’s main ideas: the safety monitoring challenge on construction sites, the AI‑driven solution architecture, its specific detection functions, and the practical advantages it offers.

## Cell 18 — 📚 Study mode (flashcards + quiz)

In [30]:
# CELL 18 - Study mode
def study_mode(res, n_cards=10):
    notes = "\n".join(f"[{fmt_ts(c['start'])}] {c['title']}: {c['summary']}" for c in res["chapters"])
    out = llm_json(f"From these video notes create {n_cards} flashcards and a 5-question multiple-choice quiz. Use ONLY the notes. "
                   'Return JSON {"flashcards":[{"q":"","a":""}],"quiz":[{"q":"","options":["A) ..","B) ..","C) ..","D) .."],"answer":"A","why":""}]}\n\n' + notes,
                   max_tokens=2200)
    L = ["## Flashcards"] + [f"**Q{i+1}.** {c['q']}  \n*A:* {c['a']}\n" for i, c in enumerate(out.get("flashcards", []))]
    L.append("## Quiz")
    for i, q in enumerate(out.get("quiz", [])):
        L.append(f"**{i+1}. {q['q']}**\n" + "\n".join(f"- {o}" for o in q.get("options", [])) + f"\n\n*Answer: {q.get('answer')}* - {q.get('why', '')}\n")
    return "\n".join(L)

STUDY_MD = study_mode(RESULT)
display(Markdown(STUDY_MD))

## Flashcards
**Q1.** What safety hazards does the AI‑based construction site surveillance system detect?  
*A:* PPE violations, fire, smoke, fights, and fallen workers.

**Q2.** Which tool normalizes video input before analysis?  
*A:* FFmpeg.

**Q3.** Which object‑detection model family is used in the system?  
*A:* YOLOv8 models.

**Q4.** What additional detection component complements the YOLOv8 models?  
*A:* A heuristic structural anomaly detector.

**Q5.** What visual information does the web‑based dashboard provide?  
*A:* Detections with bounding boxes, headcounts, and confidence scores.

**Q6.** How does the alerting module prevent email spamming?  
*A:* By applying a cooldown period between alerts.

**Q7.** Who is the primary intended user of this surveillance solution?  
*A:* Site safety officers.

**Q8.** What three main goals does the system aim to achieve?  
*A:* Improve response times, increase reliability, and enhance scalability.

**Q9.** In what format are alerts delivered to users?  
*A:* Annotated emails containing detection details.

**Q10.** How are detections represented in the system output?  
*A:* As bounding boxes with labels and alerts.

## Quiz
**1. Which component processes video frames to detect objects?**
- A) FFmpeg
- B) YOLOv8 models
- C) Dashboard
- D) Email module

*Answer: B* - YOLOv8 models perform the object‑detection step on each video frame.

**2. What is the purpose of the cooldown in the alerting module?**
- A) Increase detection speed
- B) Reduce false positives
- C) Avoid spamming recipients
- D) Enhance video quality

*Answer: C* - The cooldown limits how frequently annotated emails are sent, preventing spam.

**3. Which hazard is NOT explicitly mentioned as detected by the system?**
- A) PPE violations
- B) Fire
- C) Chemical leaks
- D) Fallen workers

*Answer: C* - Chemical leaks are not listed among the detected hazards in the notes.

**4. What does the dashboard display to help safety officers?**
- A) Weather forecasts
- B) Headcounts
- C) Financial reports
- D) Employee schedules

*Answer: B* - The dashboard visualizes headcounts along with detections and confidence scores.

**5. The system's scalability primarily benefits which aspect?**
- A) Number of construction sites monitored
- B) Color of PPE
- C) Size of email attachments
- D) Length of video clips

*Answer: A* - Scalability allows the solution to be deployed across many construction sites efficiently.


## Cell 19 — 💾 Export (Markdown + HTML; open the HTML and "Print → Save as PDF" for a PDF)

In [31]:
# CELL 19 - Export
import markdown as mdlib
md_text = render_markdown(RESULT, "detailed")
try: md_text += "\n\n" + STUDY_MD
except NameError: pass
base = re.sub(r"\W+", "_", RESULT["meta"]["title"])[:50] or "clipsense"
md_path, html_path = f"/content/{base}.md", f"/content/{base}.html"
open(md_path, "w", encoding="utf-8").write(md_text)
body = mdlib.markdown(md_text, extensions=["extra", "sane_lists"])
open(html_path, "w", encoding="utf-8").write(
    "<html><head><meta charset='utf-8'><style>body{font-family:system-ui,sans-serif;max-width:820px;margin:2rem auto;line-height:1.55;padding:0 1rem}"
    "blockquote{background:#f4f4f4;margin:0;padding:.4rem 1rem;border-left:4px solid #bbb}</style></head><body>" + body + "</body></html>")
print("Saved:", md_path, html_path)
try:
    from google.colab import files
    files.download(md_path); files.download(html_path)
except Exception as e:
    print("Download manually from the Files panel:", e)

Saved: /content/Slides_Presentation_mp4.md /content/Slides_Presentation_mp4.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cell 20 — 🖥️ Optional web UI (Gradio) with progress, summary and chat

Run this cell to get a public link with an input box for the URL, the summary, and a chat box.

In [ ]:
# CELL 20 - Gradio UI
import gradio as gr

def ui_run(url, level, lang):
    global RESULT
    try:
        RESULT = ORCH.run(url=url.strip(), out_lang=lang)
        return render_markdown(RESULT, level), "Done - ask questions below."
    except Exception as e:
        return f"**Error:** {e}", "Failed."

def ui_ask(q):
    try:
        return linkify(ORCH.bb.read("qa").ask(q), RESULT["meta"])
    except Exception as e:
        return f"Run a video first. ({e})"

with gr.Blocks(title="ClipSense") as demo:
    gr.Markdown("# 🎬 ClipSense\n*Don't watch it all. Understand it all.*")
    with gr.Row():
        url_in = gr.Textbox(label="YouTube URL", scale=4)
        level_in = gr.Dropdown(["tldr", "bullets", "detailed"], value="detailed", label="Summary level")
        lang_in = gr.Textbox(value="English", label="Output language")
    run_btn = gr.Button("Summarize", variant="primary")
    status = gr.Markdown()
    out = gr.Markdown()
    gr.Markdown("### Chat with the video")
    q_in = gr.Textbox(label="Your question")
    a_out = gr.Markdown()
    run_btn.click(ui_run, [url_in, level_in, lang_in], [out, status])
    q_in.submit(ui_ask, q_in, a_out)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://42a66fb6e84dbc1dfa.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[    0.0s]    Ingestion -> Orchestrator downloading video (this can take a while for long videos)...
[    8.8s]    Ingestion -> Orchestrator ready: 'What is AI? Artificial Intelligence Explained' (03:30), processing first 03:30
[    9.3s] Orchestrator -> All          video type = slide_lecture. Plan: vision budget 25 frames (sample every 8s) + ASR + audio analysis
[    9.4s]        Audio -> Fusion       audio ready: 4 emphasised windows
[   10.8s]   Transcript -> Orchestrator using official captions (80 lines)
[   10.8s]   Transcript -> Fusion       transcript ready: 10 segments, 601 words (youtube-captions)
[   15.3s]       Vision -> Orchestrator 27 visually-unique frames after dedupe
[   15.3s]       Vision -> Orchestrator analysing 25 frames with qwen3.8-27b
   [vision] rate limit - waiting 13s (retry 1/6)
   [vision] rate limit - waiting 9s (retry 1/6)
   [vision] rate limit - waiting 9s (retry 1/6)
   [vision] rate limit - waiting 21s (retry 1/6)
   [vision] rate limit - waiting 2

## 🛠️ Troubleshooting

**YouTube blocks the download** (`Sign in to confirm you're not a bot`, HTTP 403/429): YouTube blocks many Colab/datacenter IPs. Options, easiest first:
1. Download the video on your own computer and use **Cell 15** to upload it (sets `LOCAL_FILE`). Everything else works the same, but Whisper is used because captions need a YouTube id.
2. Export `cookies.txt` from a logged-in browser (extension "Get cookies.txt LOCALLY"), upload it in Cell 15 and set `COOKIES_FILE`. Use a throw-away account, not your main one.
3. Re-run later or from another Colab runtime.

**Captions fail but video downloads:** that is fine, the Transcript agent falls back to Groq Whisper automatically.

**Groq rate limits (HTTP 429):** the notebook waits and retries automatically. Free-tier Whisper has an hourly cap on audio seconds, so if you hit a long wait or "quota exhausted", lower `CFG["MAX_MINUTES"]`, set `LLM_WORKERS=1`, or retry in an hour. Check your limits at https://console.groq.com/settings/limits.

**Model not found (400/404):** Run Cell 3; it lists what your account has. Update `CFG[...]` accordingly. Groq changes preview models (such as the vision model) from time to time.

**Vision returns empty text:** 480p can be too small for dense slides. Set `CFG["VIDEO_HEIGHT"]=720` (then delete `/content/clipsense` to re-download).

**Respect YouTube's Terms of Service and copyright;** use this for personal study/research.

### How this maps to your proposal
Everything in the PDF's MVP is implemented, plus several stretch goals (study mode, multi-language output, export). Not included to stay 100% free and token-free: speaker diarization (pyannote needs a HuggingFace token) and LangGraph. The Orchestrator, Blackboard and message bus provide the same architecture in plain Python, and the Critic to Summarizer feedback loop is implemented in Cell 11.